# 2주차. 데이터 전처리

본 과제에서는 1주차 탐색적 데이터 분석(EDA)을 통해 확인한
데이터의 특성을 바탕으로 데이터 전처리를 수행한다.

전처리 과정에서는 결측치 처리, 이상치 처리, 인코딩,
Scaling을 수행하고 각 과정의 적용 이유와 결과를 분석한다.

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
getwd()
train <- read.csv("train.csv")
test <- read.csv("test.csv")

[1] "/Users/snagmni"

dim(train)
dim(test)

## 2. 결측치 처리

결측치는 데이터 분석 및 머신러닝 모델 학습 과정에서 문제를 일으킬 수 있으므로,
각 변수에 결측치가 존재하는지 확인한다.

In [ ]:
colSums(is.na(train))

In [ ]:
colSums(is.na(test))

In [3]:
sum(is.na(train))
sum(is.na(test))

[1] 0

[1] 0

In [4]:
num_cols <- c(
  "Age",
  "Nodule_Size",
  "TSH_Result",
  "T4_Result",
  "T3_Result"
)

In [5]:
# 각 수치형 변수의 요약 통계량 확인
summary(train[num_cols])

      Age         Nodule_Size      TSH_Result       T4_Result     
 Min.   :14.00   Min.   :0.000   Min.   : 0.100   Min.   : 4.500  
 1st Qu.:32.00   1st Qu.:1.270   1st Qu.: 2.583   1st Qu.: 6.372  
 Median :51.00   Median :2.521   Median : 5.059   Median : 8.237  
 Mean   :50.86   Mean   :2.508   Mean   : 5.057   Mean   : 8.249  
 3rd Qu.:70.00   3rd Qu.:3.761   3rd Qu.: 7.542   3rd Qu.:10.127  
 Max.   :88.00   Max.   :5.000   Max.   :10.000   Max.   :12.000  
   T3_Result    
 Min.   :0.500  
 1st Qu.:1.255  
 Median :2.004  
 Mean   :2.005  
 3rd Qu.:2.758  
 Max.   :3.500  

## 3. 이상치 처리

In [6]:
# IQR을 이용한 이상치 개수 확인

outlier_count <- sapply(train[num_cols], function(x) {
  Q1 <- quantile(x, 0.25)
  Q3 <- quantile(x, 0.75)
  IQR_value <- Q3 - Q1
  
  lower <- Q1 - 1.5 * IQR_value
  upper <- Q3 + 1.5 * IQR_value
  
  sum(x < lower | x > upper)
})

outlier_count

Age Nodule_Size  TSH_Result   T4_Result   T3_Result 
          0           0           0           0           0

In [7]:
str(train)

'data.frame':	87159 obs. of  16 variables:
 $ ID               : chr  "TRAIN_00000" "TRAIN_00001" "TRAIN_00002" "TRAIN_00003" ...
 $ Age              : int  80 37 71 40 53 86 65 36 67 58 ...
 $ Gender           : chr  "M" "M" "M" "F" ...
 $ Country          : chr  "CHN" "NGA" "CHN" "IND" ...
 $ Race             : chr  "ASN" "ASN" "MDE" "HSP" ...
 $ Family_Background: chr  "Positive" "Positive" "Positive" "Negative" ...
 $ Radiation_History: chr  "Exposed" "Unexposed" "Unexposed" "Unexposed" ...
 $ Iodine_Deficiency: chr  "Sufficient" "Sufficient" "Sufficient" "Sufficient" ...
 $ Smoke            : chr  "Non-Smoker" "Smoker" "Non-Smoker" "Non-Smoker" ...
 $ Weight_Risk      : chr  "Not Obese" "Obese" "Not Obese" "Obese" ...
 $ Diabetes         : chr  "No" "No" "Yes" "No" ...
 $ Nodule_Size      : num  0.65 2.95 2.2 3.37 4.23 ...
 $ TSH_Result       : num  2.785 0.912 0.718 6.846 0.44 ...
 $ T4_Result        : num  6.74 7.3 11.14 10.18 7.19 ...
 $ T3_Result        : num  2.576 2.505 2.38

In [8]:
categorical_cols <- c(
  "Gender",
  "Country",
  "Race",
  "Family_Background",
  "Radiation_History",
  "Iodine_Deficiency",
  "Smoke",
  "Weight_Risk",
  "Diabetes"
)

In [9]:
lapply(train[categorical_cols], unique)

$Gender
[1] "M" "F"

$Country
 [1] "CHN" "NGA" "IND" "USA" "GBR" "BRA" "RUS" "JPN" "KOR" "DEU"

$Race
[1] "ASN" "MDE" "HSP" "CAU" "AFR"

$Family_Background
[1] "Positive" "Negative"

$Radiation_History
[1] "Exposed"   "Unexposed"

$Iodine_Deficiency
[1] "Sufficient" "Deficient" 

$Smoke
[1] "Non-Smoker" "Smoker"    

$Weight_Risk
[1] "Not Obese" "Obese"    

$Diabetes
[1] "No"  "Yes"

In [10]:
sapply(train[categorical_cols], function(x) length(unique(x)))

Gender           Country              Race Family_Background 
                2                10                 5                 2 
Radiation_History Iodine_Deficiency             Smoke       Weight_Risk 
                2                 2                 2                 2 
         Diabetes 
                2

In [11]:
sapply(train[categorical_cols], function(x) length(unique(x)))

Gender           Country              Race Family_Background 
                2                10                 5                 2 
Radiation_History Iodine_Deficiency             Smoke       Weight_Risk 
                2                 2                 2                 2 
         Diabetes 
                2

In [12]:
lapply(categorical_cols, function(col) {
  setdiff(unique(test[[col]]), unique(train[[col]]))
})

[[1]]
character(0)

[[2]]
character(0)

[[3]]
character(0)

[[4]]
character(0)

[[5]]
character(0)

[[6]]
character(0)

[[7]]
character(0)

[[8]]
character(0)

[[9]]
character(0)

In [13]:
# One-Hot Encoding에 사용할 범주형 변수
categorical_cols

[1] "Gender"            "Country"           "Race"             
[4] "Family_Background" "Radiation_History" "Iodine_Deficiency"
[7] "Smoke"             "Weight_Risk"       "Diabetes"

In [14]:
# Train과 Test의 범주형 변수를 factor로 변환
train[categorical_cols] <- lapply(train[categorical_cols], factor)
test[categorical_cols] <- lapply(test[categorical_cols], factor)

# factor 변환 확인
str(train[categorical_cols])

'data.frame':	87159 obs. of  9 variables:
 $ Gender           : Factor w/ 2 levels "F","M": 2 2 2 1 1 1 1 2 2 1 ...
 $ Country          : Factor w/ 10 levels "BRA","CHN","DEU",..: 2 8 2 5 2 10 4 5 8 1 ...
 $ Race             : Factor w/ 5 levels "AFR","ASN","CAU",..: 2 2 5 4 3 1 1 3 4 4 ...
 $ Family_Background: Factor w/ 2 levels "Negative","Positive": 2 2 2 1 1 1 2 2 1 1 ...
 $ Radiation_History: Factor w/ 2 levels "Exposed","Unexposed": 1 2 2 2 2 2 2 2 1 2 ...
 $ Iodine_Deficiency: Factor w/ 2 levels "Deficient","Sufficient": 2 2 2 2 2 2 2 2 1 2 ...
 $ Smoke            : Factor w/ 2 levels "Non-Smoker","Smoker": 1 2 1 1 1 2 1 1 1 2 ...
 $ Weight_Risk      : Factor w/ 2 levels "Not Obese","Obese": 1 2 1 2 1 1 1 1 1 1 ...
 $ Diabetes         : Factor w/ 2 levels "No","Yes": 1 1 2 1 1 1 1 1 1 1 ...


In [15]:
# 범주형 변수에 One-Hot Encoding 적용
encoded_train <- model.matrix(
  ~ . - 1,
  data = train[categorical_cols]
)

# 데이터 구조 확인
dim(encoded_train)
head(encoded_train)

[1] 87159    21

,GenderF,GenderM,CountryCHN,CountryDEU,CountryGBR,CountryIND,CountryJPN,CountryKOR,CountryNGA,CountryRUS,⋯,RaceASN,RaceCAU,RaceHSP,RaceMDE,Family_BackgroundPositive,Radiation_HistoryUnexposed,Iodine_DeficiencySufficient,SmokeSmoker,Weight_RiskObese,DiabetesYes
1,0,1,1,0,0,0,0,0,0,0,⋯,1,0,0,0,1,0,1,0,0,0
2,0,1,0,0,0,0,0,0,1,0,⋯,1,0,0,0,1,1,1,1,1,0
3,0,1,1,0,0,0,0,0,0,0,⋯,0,0,0,1,1,1,1,0,0,1
4,1,0,0,0,0,1,0,0,0,0,⋯,0,0,1,0,0,1,1,0,1,0
5,1,0,1,0,0,0,0,0,0,0,⋯,0,1,0,0,0,1,1,0,0,0
6,1,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,1,1,1,0,0


In [16]:
dim(encoded_train)

[1] 87159    21

In [17]:
encoded_test <- model.matrix(
  ~ . - 1,
  data = test[categorical_cols]
)

dim(encoded_test)

[1] 46204    21

In [18]:
identical(colnames(encoded_train), colnames(encoded_test))

[1] TRUE

In [19]:
# 수치형 변수
num_cols <- c(
  "Age",
  "Nodule_Size",
  "TSH_Result",
  "T4_Result",
  "T3_Result"
)

# Train 데이터의 평균과 표준편차 저장
train_means <- sapply(train[num_cols], mean)
train_sds <- sapply(train[num_cols], sd)

# Train 데이터 표준화
scaled_train_num <- scale(
  train[num_cols],
  center = train_means,
  scale = train_sds
)

# 결과 확인
head(scaled_train_num)

Age,Nodule_Size,TSH_Result,T4_Result,T3_Result
1.34665084,-1.2883744,-0.7941119,-0.6944648,0.6583729
-0.64053072,0.3067641,-1.4487644,-0.4365200,0.5770558
0.93072911,-0.2136550,-1.5165220,1.3336551,0.4337626
-0.50189015,0.5982946,0.6254335,0.8894185,-1.4440147
0.09888567,1.1941995,-1.6137652,-0.4867768,-1.6558535
1.62393198,-0.1228544,1.3691925,0.6058861,-1.6872210


In [20]:
# Test 데이터 표준화
scaled_test_num <- scale(
  test[num_cols],
  center = train_means,
  scale = train_sds
)

# 결과 확인
head(scaled_test_num)

Age,Nodule_Size,TSH_Result,T4_Result,T3_Result
0.09888567,0.3000010,0.4814817,0.4520653,0.8406378
-0.22460900,0.6533133,-0.1144041,-1.0670172,-1.4186529
1.25422379,1.5067477,0.2120081,1.0190184,-0.8593947
0.83830206,1.2292498,0.8447001,0.2477125,0.9470861
1.20801026,0.6047429,0.7140692,0.2254260,1.6707490
0.28373977,1.1807821,1.0206968,-1.6251166,-1.1349051


In [21]:
# Train Scaling 결과의 평균과 표준편차 확인
colMeans(scaled_train_num)
apply(scaled_train_num, 2, sd)

Age   Nodule_Size    TSH_Result     T4_Result     T3_Result 
 1.371974e-16 -2.157776e-16  1.715694e-16 -2.802517e-16  1.087893e-16

Age Nodule_Size  TSH_Result   T4_Result   T3_Result 
          1           1           1           1           1

In [22]:
# 최종 전처리 Train 데이터 생성
train_processed <- data.frame(
  scaled_train_num,
  encoded_train,
  Cancer = train$Cancer
)

# 최종 전처리 Test 데이터 생성
test_processed <- data.frame(
  scaled_test_num,
  encoded_test
)

# 데이터 크기 확인
dim(train_processed)
dim(test_processed)

[1] 87159    27

[1] 46204    26

In [23]:
head(train_processed)
str(train_processed)
colSums(is.na(train_processed))

,Age,Nodule_Size,TSH_Result,T4_Result,T3_Result,GenderF,GenderM,CountryCHN,CountryDEU,CountryGBR,⋯,RaceCAU,RaceHSP,RaceMDE,Family_BackgroundPositive,Radiation_HistoryUnexposed,Iodine_DeficiencySufficient,SmokeSmoker,Weight_RiskObese,DiabetesYes,Cancer
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,1.34665084,-1.2883744,-0.7941119,-0.6944648,0.6583729,0,1,1,0,0,⋯,0,0,0,1,0,1,0,0,0,1
2,-0.64053072,0.3067641,-1.4487644,-0.4365200,0.5770558,0,1,0,0,0,⋯,0,0,0,1,1,1,1,1,0,1
3,0.93072911,-0.2136550,-1.5165220,1.3336551,0.4337626,0,1,1,0,0,⋯,0,0,1,1,1,1,0,0,1,0
4,-0.50189015,0.5982946,0.6254335,0.8894185,-1.4440147,1,0,0,0,0,⋯,0,1,0,0,1,1,0,1,0,0
5,0.09888567,1.1941995,-1.6137652,-0.4867768,-1.6558535,1,0,1,0,0,⋯,1,0,0,0,1,1,0,0,0,1
6,1.62393198,-0.1228544,1.3691925,0.6058861,-1.6872210,1,0,0,0,0,⋯,0,0,0,0,1,1,1,0,0,0


'data.frame':	87159 obs. of  27 variables:
 $ Age                        : num  1.3467 -0.6405 0.9307 -0.5019 0.0989 ...
 $ Nodule_Size                : num  -1.288 0.307 -0.214 0.598 1.194 ...
 $ TSH_Result                 : num  -0.794 -1.449 -1.517 0.625 -1.614 ...
 $ T4_Result                  : num  -0.694 -0.437 1.334 0.889 -0.487 ...
 $ T3_Result                  : num  0.658 0.577 0.434 -1.444 -1.656 ...
 $ GenderF                    : num  0 0 0 1 1 1 1 0 0 1 ...
 $ GenderM                    : num  1 1 1 0 0 0 0 1 1 0 ...
 $ CountryCHN                 : num  1 0 1 0 1 0 0 0 0 0 ...
 $ CountryDEU                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryGBR                 : num  0 0 0 0 0 0 1 0 0 0 ...
 $ CountryIND                 : num  0 0 0 1 0 0 0 1 0 0 ...
 $ CountryJPN                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryKOR                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryNGA                 : num  0 1 0 0 0 0 0 0 1 0 ...
 $ CountryRUS                 : num

Age                 Nodule_Size 
                          0                           0 
                 TSH_Result                   T4_Result 
                          0                           0 
                  T3_Result                     GenderF 
                          0                           0 
                    GenderM                  CountryCHN 
                          0                           0 
                 CountryDEU                  CountryGBR 
                          0                           0 
                 CountryIND                  CountryJPN 
                          0                           0 
                 CountryKOR                  CountryNGA 
                          0                           0 
                 CountryRUS                  CountryUSA 
                          0                           0 
                    RaceASN                     RaceCAU 
                          0                           0 
                    RaceHSP                     RaceMDE 
                          0                           0 
  Family_BackgroundPositive  Radiation_HistoryUnexposed 
                          0                           0 
Iodine_DeficiencySufficient                 SmokeSmoker 
                          0                           0 
           Weight_RiskObese                 DiabetesYes 
                          0                           0 
                     Cancer 
                          0

# 3주차. 모델 설계 및 학습, 평가

2주차 전처리 결과를 바탕으로 머신러닝 모델을 설계하고 학습한 뒤 성능을 평가한다.

## 3-1. 머신러닝 이론

### 머신러닝이란?
머신러닝은 데이터에서 패턴과 규칙을 학습하여 새로운 데이터에 대한 예측이나 판단을 수행하는 방법이다.

### 분류와 회귀
- **분류(Classification)**: 입력 데이터를 미리 정해진 범주로 분류하는 문제이다.
- **회귀(Regression)**: 연속적인 수치 값을 예측하는 문제이다.

이번 과제의 `Cancer` 변수는 0과 1로 구성된 범주형 목표변수이므로 **이진 분류 문제**에 해당한다.

### 대표적인 머신러닝 모델
1. **Logistic Regression**: 입력 변수와 목표변수의 관계를 이용하여 특정 클래스에 속할 확률을 예측하는 분류 모델이다.
2. **Decision Tree**: 데이터를 조건에 따라 반복적으로 분할하여 의사결정 규칙을 만드는 모델이다.
3. **Random Forest**: 여러 개의 결정 트리를 결합하여 예측 성능과 안정성을 높이는 앙상블 모델이다.

## 3-2. 데이터 분할

`train_processed`를 학습 데이터와 검증 데이터로 나눈다. 원래 제공된 `test_processed`에는 `Cancer` 정답값이 없으므로 모델 성능 평가에는 사용하지 않고, 최종 예측에 사용할 수 있도록 남겨둔다.

In [24]:
# 재현 가능한 결과를 위해 난수 시드를 고정한다.
set.seed(42)

# Cancer 클래스별로 80%를 학습 데이터로 선택한다.
class_indices <- split(
  seq_len(nrow(train_processed)),
  train_processed$Cancer
)

train_indices <- unlist(
  lapply(class_indices, function(idx) {
    sample(idx, size = floor(length(idx) * 0.8))
  })
)

model_train <- train_processed[train_indices, ]
model_test <- train_processed[-train_indices, ]

dim(model_train)
dim(model_test)

[1] 69727    27

[1] 17432    27

In [25]:
# 학습/검증 데이터의 클래스 분포 확인
table(model_train$Cancer)
table(model_test$Cancer)

prop.table(table(model_train$Cancer))
prop.table(table(model_test$Cancer))


    0     1 
61360  8367 


    0     1 
15340  2092 


        0         1 
0.8800034 0.1199966 


        0         1 
0.8799908 0.1200092 

In [26]:
## 3-3. 모델 학습

logistic_model <- glm(
  Cancer ~ . - 1, # 절편을 제외하고 모델을 학습(절편을 추가하면 변수간에 중복 관계 발생 가능)
  data = model_train,
  family = binomial
)

summary(logistic_model)


Call:
glm(formula = Cancer ~ . - 1, family = binomial, data = model_train)

Coefficients:
                             Estimate Std. Error z value Pr(>|z|)    
Age                          0.005274   0.011999   0.440   0.6603    
Nodule_Size                  0.010512   0.011980   0.877   0.3802    
TSH_Result                  -0.025813   0.011992  -2.152   0.0314 *  
T4_Result                    0.005628   0.011993   0.469   0.6389    
T3_Result                   -0.009252   0.012014  -0.770   0.4412    
GenderF                     -1.345643   0.057473 -23.413   <2e-16 ***
GenderM                     -1.331256   0.058444 -22.778   <2e-16 ***
CountryCHN                  -0.037704   0.051031  -0.739   0.4600    
CountryDEU                  -0.040289   0.068723  -0.586   0.5577    
CountryGBR                  -0.008640   0.067970  -0.127   0.8989    
CountryIND                   0.705701   0.045911  15.371   <2e-16 ***
CountryJPN                  -0.033617   0.059814  -0.562   0.5741    

In [ ]:
### 모델 선택 및 설정

본 과제에서는 이진 분류 문제인 `Cancer`를 예측하기 위해 Logistic Regression을 선택하였다. Logistic Regression은 각 관측치가 Cancer=1에 속할 확률을 예측할 수 있으며, 변수별 계수와 통계적 유의성을 확인할 수 있다는 장점이 있다.

이번 모델에서는 `family = binomial`을 사용하여 이진 분류를 수행하였다. 또한 2주차에서 모든 범주형 변수의 One-Hot Encoding을 수행하였기 때문에 절편을 제외하기 위해 `Cancer ~ . - 1`을 사용하였다.

예측 확률이 0.5 이상이면 Cancer=1, 0.5 미만이면 Cancer=0으로 분류하였다.

In [27]:
## 3-4. 모델 평가

# Cancer = 1일 확률 예측
pred_prob <- predict(
  logistic_model,
  newdata = model_test,
  type = "response"
)

# 확률이 0.5 이상이면 Cancer = 1로 분류
pred_class <- ifelse(pred_prob >= 0.5, 1, 0)

# 실제값과 예측값 비교
confusion_matrix <- table(
  Actual = model_test$Cancer,
  Predicted = pred_class
)

confusion_matrix

      Predicted
Actual     0     1
     0 15322    18
     1  2072    20

In [28]:
## 평가 지표 계산

TN <- confusion_matrix[1, 1]
FP <- confusion_matrix[1, 2]
FN <- confusion_matrix[2, 1]
TP <- confusion_matrix[2, 2]

accuracy <- (TP + TN) / (TP + TN + FP + FN)
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
f1_score <- 2 * precision * recall / (precision + recall)

accuracy
precision
recall
f1_score

[1] 0.8801056

[1] 0.5263158

[1] 0.009560229

[1] 0.01877934

In [29]:
data.frame(
  Accuracy = accuracy,
  Precision = precision,
  Recall = recall,
  F1_Score = f1_score
)

Accuracy,Precision,Recall,F1_Score
<dbl>,<dbl>,<dbl>,<dbl>
0.8801056,0.5263158,0.009560229,0.01877934


In [30]:
## 오분류 분석

misclassified <- model_test[
  model_test$Cancer != pred_class,
]

dim(misclassified)

table(
  Actual = misclassified$Cancer,
  Predicted = pred_class[
    model_test$Cancer != pred_class
  ]
)

[1] 2090   27

      Predicted
Actual    0    1
     0    0   18
     1 2072    0

In [31]:
## False Negative 분석

false_negative <- model_test[
  model_test$Cancer == 1 & pred_class == 0,
]

dim(false_negative)

[1] 2072   27

In [32]:
head(false_negative)

,Age,Nodule_Size,TSH_Result,T4_Result,T3_Result,GenderF,GenderM,CountryCHN,CountryDEU,CountryGBR,⋯,RaceCAU,RaceHSP,RaceMDE,Family_BackgroundPositive,Radiation_HistoryUnexposed,Iodine_DeficiencySufficient,SmokeSmoker,Weight_RiskObese,DiabetesYes,Cancer
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
5,0.09888567,1.1941995,-1.6137652,-0.4867768,-1.655853548,1,0,1,0,0,⋯,1,0,0,0,1,1,0,0,0,1
9,0.74587502,1.4439554,0.6134635,-1.5929440,-0.003192842,0,1,0,0,0,⋯,0,1,0,0,0,0,0,0,0,1
38,-1.28752007,1.5970587,-0.6604739,0.7335630,0.348158369,0,1,0,0,0,⋯,0,1,0,0,1,0,0,0,1,1
53,-1.37994712,-0.3795662,0.2044837,0.3762694,-1.503302168,1,0,0,1,0,⋯,0,0,0,1,0,0,0,0,1,1
81,-1.51858769,-0.8030145,0.6744086,0.5189696,-1.032914654,0,1,1,0,0,⋯,0,0,0,1,0,1,0,1,0,1
116,0.42238034,-1.1427909,-0.2855927,0.7359532,0.239820869,1,0,0,0,0,⋯,0,0,1,1,1,1,1,0,0,1


In [33]:
## False Negative의 원본 데이터 확인

# model_test에 해당하는 원본 train 행 번호
model_test_indices <- setdiff(
  seq_len(nrow(train_processed)),
  train_indices
)

# False Negative의 model_test 내부 위치
fn_positions <- which(
  model_test$Cancer == 1 & pred_class == 0
)

# 원본 train 데이터의 행 번호로 변환
fn_original_indices <- model_test_indices[fn_positions]

# 원본 데이터에서 False Negative 추출
false_negative_original <- train[fn_original_indices, ]

dim(false_negative_original)
head(false_negative_original)

[1] 2072   16

,ID,Age,Gender,Country,Race,Family_Background,Radiation_History,Iodine_Deficiency,Smoke,Weight_Risk,Diabetes,Nodule_Size,TSH_Result,T4_Result,T3_Result,Cancer
,<chr>,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
5,TRAIN_00004,53,F,CHN,CAU,Negative,Unexposed,Sufficient,Non-Smoker,Not Obese,No,4.230048,0.439519,7.194450,0.569356,1
9,TRAIN_00008,67,M,NGA,HSP,Negative,Exposed,Deficient,Non-Smoker,Not Obese,No,4.590178,6.812131,4.798520,2.002234,1
38,TRAIN_00037,23,M,IND,HSP,Negative,Unexposed,Deficient,Non-Smoker,Not Obese,Yes,4.810942,3.167104,9.837675,2.306860,1
53,TRAIN_00052,21,F,DEU,ASN,Positive,Exposed,Deficient,Non-Smoker,Not Obese,Yes,1.960791,5.641946,9.063786,0.701620,1
81,TRAIN_00080,18,M,CHN,ASN,Positive,Exposed,Sufficient,Non-Smoker,Obese,No,1.350209,6.986509,9.372871,1.109452,1
116,TRAIN_00115,60,F,RUS,MDE,Positive,Unexposed,Sufficient,Smoker,Not Obese,No,0.860276,4.239725,9.842852,2.212930,1


In [34]:
## False Negative 특성 분석

# 수치형 변수 평균
colMeans(
  false_negative_original[, c(
    "Age",
    "Nodule_Size",
    "TSH_Result",
    "T4_Result",
    "T3_Result"
  )]
)

# 범주형 변수별 분포
table(false_negative_original$Gender)
table(false_negative_original$Family_Background)
table(false_negative_original$Radiation_History)
table(false_negative_original$Iodine_Deficiency)
table(false_negative_original$Smoke)
table(false_negative_original$Weight_Risk)
table(false_negative_original$Diabetes)

Age Nodule_Size  TSH_Result   T4_Result   T3_Result 
  50.124035    2.514610    5.062992    8.290005    1.998532


   F    M 
1235  837 


Negative Positive 
    1263      809 


  Exposed Unexposed 
      432      1640 


 Deficient Sufficient 
       703       1369 


Non-Smoker     Smoker 
      1685        387 


Not Obese     Obese 
     1447       625 


  No  Yes 
1681  391 

In [35]:
## 실제 Cancer=1 전체와 False Negative 비교

actual_positive <- train[train$Cancer == 1, ]

data.frame(
  Group = c("전체 실제 암", "False Negative"),
  N = c(
    nrow(actual_positive),
    nrow(false_negative_original)
  ),
  Mean_Age = c(
    mean(actual_positive$Age),
    mean(false_negative_original$Age)
  ),
  Mean_Nodule_Size = c(
    mean(actual_positive$Nodule_Size),
    mean(false_negative_original$Nodule_Size)
  ),
  Mean_TSH = c(
    mean(actual_positive$TSH_Result),
    mean(false_negative_original$TSH_Result)
  ),
  Mean_T4 = c(
    mean(actual_positive$T4_Result),
    mean(false_negative_original$T4_Result)
  ),
  Mean_T3 = c(
    mean(actual_positive$T3_Result),
    mean(false_negative_original$T3_Result)
  )
)

Group,N,Mean_Age,Mean_Nodule_Size,Mean_TSH,Mean_T4,Mean_T3
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
전체 실제 암,10459,50.80916,2.512292,4.993172,8.268170,1.996707
False Negative,2072,50.12403,2.514610,5.062992,8.290005,1.998532


## 오분류 분석 결론

Logistic Regression 모델의 오분류를 분석한 결과, 총 2,090건의 오분류 중 2,072건이 실제 암 환자를 정상으로 예측한 False Negative로 나타났다. 따라서 본 모델은 암 환자를 탐지하는 Recall 성능이 매우 낮은 것으로 확인되었다.

False Negative 집단과 전체 실제 암 환자 집단의 평균을 비교한 결과, Age, Nodule_Size, TSH_Result, T4_Result, T3_Result 등의 수치형 변수에서 두 집단의 평균값이 전반적으로 유사하게 나타났다. 따라서 이번 분석만으로는 특정 수치형 변수가 False Negative 발생의 주요 원인이라고 판단하기 어려웠다.

또한 False Negative 집단은 성별, 가족력, 방사선 노출 여부, 요오드 결핍 여부, 흡연 여부, 비만 여부, 당뇨 여부 등 여러 범주에 걸쳐 다양하게 분포하였다. 이를 통해 특정 하나의 변수나 범주만으로 오분류를 설명하기는 어려웠다.

이번 분석을 통해 클래스 불균형이 존재하는 분류 문제에서는 Accuracy만으로 모델의 성능을 판단하기 어렵다는 점을 확인하였다. 특히 암 분류에서는 실제 암 환자를 놓치는 False Negative가 중요하므로 Recall과 F1-score를 함께 고려할 필요가 있다.

### Random Forest 모델 추가 선택 이유

앞서 Logistic Regression을 이용하여 갑상선암 분류 모델을 구축한 결과, Accuracy는 88.01%로 비교적 높게 나타났다. 그러나 Recall은 0.96%, F1-score는 1.88%로 매우 낮았으며, 실제 암 환자 2,092명 중 2,072명을 정상으로 예측하는 False Negative가 발생하였다.

이는 본 데이터에서 정상 클래스와 암 클래스 사이에 불균형이 존재하고 있으며, Logistic Regression만으로는 현재 데이터의 복잡한 패턴을 충분히 학습하지 못했을 가능성을 보여준다.

따라서 Logistic Regression과 다른 방식으로 데이터를 학습하는 모델을 추가하여 성능을 비교하고자 Random Forest를 선택하였다.

Random Forest는 여러 개의 Decision Tree를 결합하는 앙상블 모델로, Logistic Regression과 달리 변수와 Target 사이의 비선형적인 관계를 학습할 수 있다는 장점이 있다. 또한 여러 변수를 함께 고려하는 과정에서 변수 간의 상호작용을 반영할 수 있기 때문에, 기존 Logistic Regression에서 놓친 패턴을 학습할 수 있는지 확인하기에 적합하다고 판단하였다.

따라서 Random Forest를 추가로 학습한 후 Accuracy, Precision, Recall, F1-score와 Confusion Matrix를 비교하여 두 모델 중 갑상선암 분류에 더 적합한 모델을 확인하고자 한다.

In [44]:
library(randomForest)

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


다음의 패키지를 부착합니다: ‘randomForest’


The following object is masked from ‘package:dplyr’:

    combine


The following object is masked from ‘package:ggplot2’:

    margin




In [45]:
str(model_train)

'data.frame':	69727 obs. of  27 variables:
 $ Age                        : num  0.885 -0.132 0.838 -1.334 -0.456 ...
 $ Nodule_Size                : num  -0.109 1.5 -0.394 1.209 -0.851 ...
 $ TSH_Result                 : num  0.00872 -1.70493 -0.05927 0.31408 0.38248 ...
 $ T4_Result                  : num  1.62 -0.15 1.222 0.849 0.452 ...
 $ T3_Result                  : num  0.245 -1.254 -0.101 0.954 -1.572 ...
 $ GenderF                    : num  1 1 1 0 0 1 1 1 1 0 ...
 $ GenderM                    : num  0 0 0 1 1 0 0 0 0 1 ...
 $ CountryCHN                 : num  0 0 0 0 0 0 1 0 1 0 ...
 $ CountryDEU                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryGBR                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryIND                 : num  0 0 0 0 1 0 0 1 0 0 ...
 $ CountryJPN                 : num  0 0 0 1 0 1 0 0 0 0 ...
 $ CountryKOR                 : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CountryNGA                 : num  1 0 0 0 0 0 0 0 0 0 ...
 $ CountryRUS                 : num

In [46]:
rf_train <- model_train
rf_test <- model_test

rf_train$Cancer <- as.factor(rf_train$Cancer)
rf_test$Cancer <- as.factor(rf_test$Cancer)
str(rf_train$Cancer)

 Factor w/ 2 levels "0","1": 1 1 1 1 1 1 1 1 1 1 ...


In [47]:
set.seed(42)

rf_model <- randomForest(
  Cancer ~ .,
  data = rf_train,
  ntree = 100,
  mtry = 5,
  importance = TRUE
)

In [48]:
rf_model


Call:
 randomForest(formula = Cancer ~ ., data = rf_train, ntree = 100,      mtry = 5, importance = TRUE) 
               Type of random forest: classification
                     Number of trees: 100
No. of variables tried at each split: 5

        OOB estimate of  error rate: 11.84%
Confusion matrix:
      0    1 class.error
0 59954 1406  0.02291395
1  6851 1516  0.81881200

In [50]:
rf_pred_class <- predict(
  rf_model,
  newdata = rf_test
)
rf_pred_class

4     5     7     8     9    14    16    24    29    34    38    52    53 
    0     0     0     0     0     0     0     0     0     0     1     0     0 
   59    65    70    79    81    82    85   101   103   110   116   117   119 
    0     0     1     0     1     0     0     0     0     0     0     0     0 
  120   123   126   127   135   139   150   151   158   169   170   174   179 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  183   196   199   211   214   216   218   219   220   221   224   225   228 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  262   267   269   271   280   291   293   297   300   304   309   310   319 
    1     0     0     0     0     0     0     0     0     0     0     0     0 
  324   325   326   332   335   338   340   347   349   351   354   355   360 
    0     0     0     0     0     0     0     0     1     0     0     0     0 
  361   365   374   376   387   388   390   392   398   404   405   426   438 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  440   444   447   460   466   467   472   478   483   498   499   500   507 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  508   509   510   515   520   522   525   527   536   540   561   564   567 
    0     0     0     0     0     1     0     0     0     0     0     0     0 
  574   575   579   580   593   597   598   603   608   609   617   624   626 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  630   639   643   649   656   658   666   673   678   680   685   687   688 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  697   713   719   729   735   741   742   745   751   756   760   762   766 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  784   788   789   791   793   799   804   810   812   817   826   828   830 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  838   852   859   864   866   868   872   873   876   894   906   914   917 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  921   922   938   941   943   953   957   958   961   965   973   981   986 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
  989   994  1008  1012  1017  1029  1031  1034  1042  1051  1058  1060  1061 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1063  1064  1065  1070  1081  1088  1094  1098  1099  1105  1109  1111  1116 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1122  1125  1135  1139  1149  1150  1151  1156  1160  1164  1165  1167  1182 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1184  1185  1186  1195  1197  1207  1210  1217  1221  1225  1226  1237  1238 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1244  1245  1251  1257  1260  1264  1266  1281  1289  1290  1298  1299  1301 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1302  1306  1320  1324  1326  1328  1337  1339  1344  1350  1360  1364  1365 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1367  1377  1380  1381  1390  1405  1419  1431  1435  1438  1444  1451  1452 
    0     0     0     0     0     0     0     0     0     0     0     0     0 
 1459  1462  1465  1467  1472  1478  1489  1500  1502  1503  1506  1512  1520 
    0     0     0     1     0     0     0     0     0     0     0     0     0 
 1527  1533  1536  1537  1544  1545  1555  1562  1563  1566  1568  1607  1618 
    1     0     0     0     0     0     0     0     0     0     0     0     0 
 1621  1651  1660  1664  1668  1673  1674  1675  1692  1695  1699  1703  1707 
    0     0     0     0     0     0     0     0     0     0     0     0     1 
 1709  1710  1711  1714  1716  1724  1725  1740  1741 

In [51]:
rf_confusion_matrix <- table(
  Actual = rf_test$Cancer,
  Predicted = rf_pred_class
)

rf_confusion_matrix

      Predicted
Actual     0     1
     0 15009   331
     1  1719   373

In [52]:
TN <- rf_confusion_matrix[1, 1]
FP <- rf_confusion_matrix[1, 2]
FN <- rf_confusion_matrix[2, 1]
TP <- rf_confusion_matrix[2, 2]

rf_accuracy <- (TP + TN) / (TP + TN + FP + FN)
rf_precision <- TP / (TP + FP)
rf_recall <- TP / (TP + FN)
rf_f1_score <- 2 * rf_precision * rf_recall /
  (rf_precision + rf_recall)

rf_accuracy
rf_precision
rf_recall
rf_f1_score

[1] 0.8824002

[1] 0.5298295

[1] 0.1782983

[1] 0.2668097

### Logistic Regression과 Random Forest 비교

두 모델의 성능을 비교한 결과, Random Forest가 Logistic Regression보다 전반적으로 개선된 성능을 보였다.

Logistic Regression의 Accuracy는 88.01%, Precision은 52.63%, Recall은 0.96%, F1-score는 1.88%로 나타났다. 특히 Recall이 매우 낮아 실제 암 환자를 대부분 정상으로 분류하는 문제가 확인되었다.

Random Forest의 경우 Accuracy는 88.24%, Precision은 52.98%, Recall은 17.83%, F1-score는 26.68%로 나타났다.

특히 Recall이 Logistic Regression의 0.96%에서 Random Forest의 17.83%로 크게 증가하였다. 실제 암 환자 2,092명 중에서도 Logistic Regression은 20명을 탐지한 반면, Random Forest는 373명을 탐지하였다.

따라서 Random Forest가 Logistic Regression보다 실제 암 환자를 탐지하는 능력이 개선된 것으로 확인되었다. 그러나 Random Forest에서도 1,719건의 False Negative가 발생하여 Recall이 17.83%에 그쳤다. 따라서 Accuracy만을 기준으로 모델을 평가하기보다는 암 분류 문제의 특성을 고려하여 Recall과 F1-score를 함께 확인할 필요가 있다.

이번 비교에서는 Random Forest를 Logistic Regression보다 더 적합한 모델로 판단할 수 있지만, 실제 암 환자를 놓치는 False Negative를 줄이기 위해서는 추가적인 모델 개선이 필요하다.